# DA-417. Аналитика по кухням

Костя, стажёр. Задача: посмотреть по кухням, кого продвигать в следующем квартале.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2


In [2]:
conn = psycopg2.connect(
    host="localhost", port=5433, dbname="casino", user="analyst", password="analyst")
conn.cursor().execute("SET search_path TO delivery")


In [7]:
# выгружаю всё, дальше режу в пандасе
df = pd.read_sql('''
SELECT *
FROM orders o
JOIN restaurants r ON r.restaurant_id = o.restaurant_id
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.created_at BETWEEN '2025-10-01' AND '2026-06-29'
  AND o.promo_code <> 'NONE'
''', conn)
df.shape


## Выручка по кухням

In [8]:
rev = df.groupby('cuisine').apply(
    lambda g: (g['items_total'] * g['commission_pct'] / 100).sum()).sort_values(ascending=False)
rev


In [9]:
rev.plot(kind='bar')
plt.title('Выручка по кухням')
plt.show()


Грузинская кухня даёт больше всех, дальше пицца. На третьем месте азиатская, суши чуть ниже.

**Вывод: продвигаем азиатскую, у неё выручка выше, чем у суши.**


## Время доставки

In [3]:
t = pd.read_sql('''
SELECT c.city,
       avg(EXTRACT(epoch FROM (o.delivered_at - o.created_at))/60) AS avg_delivery_min,
       count(*) AS n
FROM orders o
JOIN restaurants r ON r.restaurant_id = o.restaurant_id
JOIN cities c ON c.city_id = r.city_id
WHERE o.delivered_at IS NOT NULL
GROUP BY c.city
''', conn)
t


In [4]:
overall = t['avg_delivery_min'].mean()
print('среднее время доставки:', round(overall, 1), 'мин')


Обещаем 45 минут, по факту 58,8.

**Вывод: SLA по доставке провален, надо нанимать курьеров.**


## Топ ресторанов

In [5]:
top = pd.read_sql('''
SELECT r.restaurant_name, r.cuisine,
       sum(o.items_total * r.commission_pct / 100) AS revenue, r.rating
FROM orders o
JOIN restaurants r ON r.restaurant_id = o.restaurant_id
WHERE o.created_at BETWEEN '2025-10-01' AND '2026-06-29'
  AND r.rating > 4.2
GROUP BY 1, 2, 4
ORDER BY revenue DESC
LIMIT 20
''', conn)
top.head(20)


In [6]:
sample = top.sample(5)
sample


## Итог

1. Продвигаем азиатскую кухню — она обгоняет суши по выручке.
2. Нанимаем курьеров, среднее время доставки 58,8 минуты против обещанных 45.
3. Топ-20 ресторанов с рейтингом выше 4,2 — в промо.


In [ ]:
df.to_csv('/Users/kostya/Desktop/analiz/vygruzka_final_2.csv')
